<a href="https://colab.research.google.com/github/reaglin/Video-From-PowerPoint-using-Gemini/blob/main/PPTX_Create_Narration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ PPTX Narration Pipeline
### Automated AI Narration and Audio Embedding for PowerPoint Presentations

**What this notebook does:**

This pipeline takes a PowerPoint presentation and automatically:
1. Reads the slide content and speaker notes from each slide
2. Uses Google Gemini AI to write natural spoken narration for each slide
3. Converts that narration to a professional voice audio file using Google Cloud Text-to-Speech
4. Embeds the audio files directly into the PowerPoint slides with autoplay timing
5. Produces a `.pptx` file ready to export as an MP4 video

**Prerequisites — you will need:**
- A Google account with Google Drive
- A Gemini API key (free, from Google AI Studio)
- A Google Cloud account with Text-to-Speech API enabled (requires billing setup, but cost is negligible)
- A Google Cloud service account credentials JSON file

**Workflow:**

| Cell | What it does |
|------|--------------|
| Cell 1 | Mount Google Drive and install required packages |
| Cell 2 | Configuration — set your paths, API keys, and preferences |
| Cell 3 | **Step 1** — Extract slide content and generate narration text via Gemini |
| Cell 4 | Optional — Preview all generated narrations in the notebook |
| Cell 5 | **Step 2** — Generate WAV audio files via Google Cloud TTS |
| Cell 6 | **Step 3** — Embed audio into PPTX with autoplay timing |

> ⚠️ **Run cells in order, top to bottom.** Cell 1 and Cell 2 must be run at the start of every session.

---

## 📦 Cell 1 — Mount Google Drive & Install Packages

**Run this cell every time you open the notebook.**

Colab does not keep installed packages between sessions, so this must be run at the start of each session. Google Drive mounting will ask you to authorize access the first time — follow the prompts.

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted')

# Install required packages
# python-pptx         : read and manipulate PowerPoint files
# google-generativeai : Gemini AI for narration generation
# google-cloud-texttospeech : Google TTS for audio generation
!pip install -q python-pptx google-genai google-cloud-texttospeech
print('✅ Packages installed')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted
✅ Packages installed


## ⚙️ Cell 2 — Configuration

**Edit this cell before running anything else.** All settings for the pipeline are defined here.

### Folder Structure
The pipeline uses a single root folder in your Google Drive with three subfolders:
```
ROOT_PATH/
  ├── credentials/          ← your Google Cloud credentials JSON file goes here
  ├── presentations/        ← your .pptx files go here
  └── output/
      └── <presentation_name>/   ← created automatically per presentation
          ├── narrations.json    ← generated narration text (review before Step 2)
          ├── slide_01.wav
          ├── slide_02.wav
          └── <name>_narrated.pptx
```

### Getting Your API Keys

**Gemini API Key:**
1. Go to https://aistudio.google.com/apikey
2. Sign in with your Google account
3. Click 'Create API Key' and copy it
4. Paste it below as `GEMINI_API_KEY`

**Google Cloud TTS Credentials:**
1. Go to https://console.cloud.google.com
2. Create a new project (e.g. 'TTS-Narration')
3. Search for 'Cloud Text-to-Speech API' → Enable it
4. Go to IAM & Admin → Service Accounts → Create Service Account
5. Grant role: 'Cloud Text-to-Speech User'
6. Go to Keys tab → Add Key → Create new key → JSON → Download
7. Rename the downloaded file to `google_tts_credentials.json`
8. Upload it to `ROOT_PATH/credentials/` in your Google Drive

### Gemini Model Options
Use one of these models for narration generation. Run the helper cell below to see all available models on your account.

| Model | Speed | Quality | Recommended |
|-------|-------|---------|-------------|
| `models/gemini-2.5-flash` | Fast | Very Good | ✅ Best choice for most users |
| `models/gemini-2.5-pro` | Slow | Excellent | Use for highest quality, but expect 30-60s per slide |
| `models/gemini-2.0-flash-001` | Very Fast | Good | Use if 2.5-flash is unavailable |

### Google Cloud TTS Voice Options
Google offers several voice tiers. **Studio voices** are the highest quality and recommended for professional narration.

| Voice Name | Gender | Quality | Notes |
|------------|--------|---------|-------|
| `en-US-Studio-M` | Male | ⭐⭐⭐⭐⭐ | Best professional male voice |
| `en-US-Studio-O` | Female | ⭐⭐⭐⭐⭐ | Best professional female voice |
| `en-US-Neural2-D` | Male | ⭐⭐⭐⭐ | Good quality, lower cost |
| `en-US-Neural2-F` | Female | ⭐⭐⭐⭐ | Good quality, lower cost |
| `en-US-Wavenet-D` | Male | ⭐⭐⭐ | Standard quality |

To see all available voices: https://cloud.google.com/text-to-speech/docs/voices

**Speaking Rate:** 1.0 is normal speed. 0.9 is slightly slower (recommended for technical content). Range: 0.25 to 4.0.

**Pitch:** 0.0 is neutral. Negative values (e.g. -1.0) give a slightly deeper tone. Range: -20.0 to 20.0.

In [ ]:
import os
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# REQUIRED: Edit these settings before running any other cells
# ══════════════════════════════════════════════════════════════════════════════

# Root folder path in Google Drive
# Everything the pipeline needs lives under this folder
ROOT_PATH = "/content/drive/MyDrive/root_location_of_files"

# Name of your .pptx file (just the filename, not the full path)
# The file must be in ROOT_PATH/presentations/
PPTX_FILENAME = "presentation03.pptx"  # This is an example - your pptx here

# Your Gemini API key — get from https://aistudio.google.com/apikey
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"

# ══════════════════════════════════════════════════════════════════════════════
# OPTIONAL: Change these settings to customize the pipeline
# ══════════════════════════════════════════════════════════════════════════════

# Gemini model for narration generation
# Recommended: 'models/gemini-2.5-flash' (fast, high quality)
# Alternative: 'models/gemini-2.5-pro' (slower but highest quality)
GEMINI_MODEL = "models/gemini-2.5-flash"

# Google Cloud TTS voice
# Recommended: 'en-US-Studio-M' (professional male, highest quality)
# Alternative: 'en-US-Studio-O' (professional female, highest quality)
# See full voice list: https://cloud.google.com/text-to-speech/docs/voices
VOICE_NAME = "en-US-Studio-M"

# Speaking rate: 1.0 = normal, 0.95 = slightly slower (good for technical content)
SPEAKING_RATE = 1.1

# Voice pitch: 0.0 = neutral, negative = deeper, positive = higher
PITCH = 0.0

# Name of your Google Cloud credentials JSON file
# Must be placed in ROOT_PATH/credentials/
CREDENTIALS_FILENAME = "google_tts_credentials.json"

# ══════════════════════════════════════════════════════════════════════════════
# Derived paths — do not edit below this line
# ══════════════════════════════════════════════════════════════════════════════

ROOT                 = Path(ROOT_PATH)
PRESENTATIONS_DIR    = ROOT / 'presentations'
OUTPUT_DIR           = ROOT / 'output'
CREDENTIALS_DIR      = ROOT / 'credentials'
PPTX_PATH            = PRESENTATIONS_DIR / PPTX_FILENAME
CREDENTIALS_PATH     = CREDENTIALS_DIR / CREDENTIALS_FILENAME
PRESENTATION_NAME    = Path(PPTX_FILENAME).stem
SLIDE_OUTPUT_DIR     = OUTPUT_DIR / PRESENTATION_NAME
NARRATIONS_FILE      = SLIDE_OUTPUT_DIR / 'narrations.json'

# Create all required directories if they don't exist
for d in [PRESENTATIONS_DIR, OUTPUT_DIR, CREDENTIALS_DIR, SLIDE_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Set Google Cloud credentials environment variable
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(CREDENTIALS_PATH)

# ── Validate configuration ─────────────────────────────────────────────────────
errors = []
if not PPTX_PATH.exists():
    errors.append(f'❌ PPTX file not found: {PPTX_PATH}')
if not CREDENTIALS_PATH.exists():
    errors.append(f'❌ Credentials file not found: {CREDENTIALS_PATH}')
if GEMINI_API_KEY == 'YOUR_GEMINI_API_KEY':
    errors.append('❌ Please set your Gemini API key')
if PPTX_FILENAME == 'YOUR_PRESENTATION.pptx':
    errors.append('❌ Please set your PPTX filename')

if errors:
    for e in errors:
        print(e)
else:
    print('✅ Configuration valid')
    print(f'   Presentation  : {PPTX_PATH}')
    print(f'   Output folder : {SLIDE_OUTPUT_DIR}')
    print(f'   Narrations    : {NARRATIONS_FILE}')
    print(f'   Voice         : {VOICE_NAME} (rate={SPEAKING_RATE}, pitch={PITCH})')
    print(f'   Gemini model  : {GEMINI_MODEL}')

## 🔍 Cell 2b — Optional: List Available Gemini Models

Run this cell if you get a 404 error in Cell 3 — it lists all Gemini models available on your API key so you can choose the right one for `GEMINI_MODEL` in Cell 2.

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

print('Available models that support content generation:')
print('─' * 50)
for m in client.models.list():
    print(f'  {m.name}')
print()
print(f'Currently configured: {GEMINI_MODEL}')

---
## 📝 Cell 3 — Step 1: Generate Narration Text (Gemini AI)

This cell reads each slide's text content and speaker notes, then uses Gemini AI to write natural spoken narration for each slide.

**What it does:**
- Extracts all visible text and speaker notes from each slide
- Sends the content to Gemini with a prompt asking for natural spoken explanation
- Saves the result to `narrations.json` in your output folder after each slide (crash-safe)
- Skips slides that were already successfully generated (safe to re-run)

**After this cell completes:**
> ⚠️ **Open `narrations.json` in your Drive output folder and review the narration text before running Cell 5.** Edit any slides where the narration missed the point, used wrong terminology, or is too long/short. This is your quality control checkpoint and is much faster than re-generating audio.

**If you get a 404 model error:** Run Cell 2b to see available models, then update `GEMINI_MODEL` in Cell 2 and re-run Cell 2 before retrying this cell.

**Estimated time:** ~5-10 seconds per slide with `gemini-2.5-flash`, ~30-60 seconds with `gemini-2.5-pro`.

In [ ]:
# Note: python-pptx is installed in Cell 1. If you get an ImportError,
# run Cell 1 first to install all required packages.

import json
import time
from google import genai
from pptx import Presentation

# ── Gemini setup ───────────────────────────────────────────────────────────────
client = genai.Client(api_key=GEMINI_API_KEY)
#model = genai.GenerativeModel(GEMINI_MODEL)

# ── Narration prompt ───────────────────────────────────────────────────────────
# This prompt instructs Gemini how to write the narration.
# Modify the tone/style instructions here if needed for your subject matter.

# You may wish to modify this narration prompt with your specific requirements
# for your prenestation

NARRATION_PROMPT = """You are creating spoken narration for an educational course presentation slide.

Your task:
- Write a clear, direct spoken explanation of the slide content
- Explain the major concepts, terms, and ideas presented on the slide
- Use the speaker notes as guidance for what to emphasize
- Write in a natural spoken style as if a knowledgeable professor is explaining the material directly
- Do NOT say "In this slide" or "As you can see" — just explain the content directly
- Do NOT use bullet point language — write in flowing spoken sentences
- Length should be proportional to the complexity and amount of content on the slide
- For title or transition slides with minimal content, keep the narration brief (2-3 sentences)

SLIDE TITLE:
{title}

SLIDE CONTENT (text from slide):
{content}

SPEAKER NOTES (objectives and guidance):
{notes}

Write the spoken narration now:"""

# ── Helper functions ───────────────────────────────────────────────────────────
def get_slide_title(slide):
    if slide.shapes.title and slide.shapes.title.text.strip():
        return slide.shapes.title.text.strip()
    return 'Untitled Slide'

def extract_slide_text(slide):
    texts = []
    for shape in slide.shapes:
        if shape.has_text_frame:
            for para in shape.text_frame.paragraphs:
                line = para.text.strip()
                if line:
                    texts.append(line)
    return '\n'.join(texts)

def extract_notes(slide):
    if slide.has_notes_slide:
        return slide.notes_slide.notes_text_frame.text.strip()
    return ''

# ── Load existing narrations (resume from where we left off) ───────────────────
if NARRATIONS_FILE.exists():
    with open(NARRATIONS_FILE, 'r', encoding='utf-8') as f:
        narrations = json.load(f)
    print(f'Resuming — {len(narrations)} slides already generated\n')
else:
    narrations = {}
    print('Starting fresh — no existing narrations found\n')

# ── Process each slide ─────────────────────────────────────────────────────────
prs   = Presentation(PPTX_PATH)
total = len(prs.slides)
print(f'Presentation : {PPTX_PATH.name}')
print(f'Total slides : {total}')
print(f'Output file  : {NARRATIONS_FILE}\n')

for i, slide in enumerate(prs.slides, start=1):
    slide_key = f'slide_{i:02d}'

    if slide_key in narrations:
        print(f'  [{i:02d}/{total}] {slide_key} — skipped (already generated)')
        continue

    title   = get_slide_title(slide)
    content = extract_slide_text(slide)
    notes   = extract_notes(slide)

    print(f'  [{i:02d}/{total}] Generating: {title[:55]}', end='', flush=True)

    try:
        prompt   = NARRATION_PROMPT.format(
            title   = title,
            content = content or '(No text content on slide)',
            notes   = notes   or '(No speaker notes provided)'
        )
        response  = client.models.generate_content(
          model   = GEMINI_MODEL,
          contents = prompt
        )
        narration = response.text.strip()

        narrations[slide_key] = {
            'slide_number': i,
            'title'       : title,
            'narration'   : narration
        }

        # Save after every slide — if the session crashes, progress is preserved
        with open(NARRATIONS_FILE, 'w', encoding='utf-8') as f:
            json.dump(narrations, f, indent=2, ensure_ascii=False)

        words = len(narration.split())
        est_s = int(words / 2.5)  # ~150 words per minute
        print(f' ✓  ({words} words, ~{est_s}s audio)')

    except Exception as e:
        print(f'\n       ✗ ERROR: {e}')
        narrations[slide_key] = {
            'slide_number': i,
            'title'       : title,
            'narration'   : f'[GENERATION FAILED: {e}]'
        }

    if i < total:
        time.sleep(2)  # brief pause between API calls

print(f'\n✅ Step 1 complete!')
print(f'   Narrations saved to: {NARRATIONS_FILE}')
print()
print('⚠️  NEXT STEP: Review narrations.json in your Drive output folder.')
print('   Edit any narration text that needs correction before running Cell 5.')
print('   This is your quality control checkpoint.')

---
## 🔍 Cell 4 — Optional: Preview Narrations in Notebook

Prints all generated narrations directly in the notebook so you can review them without opening the JSON file. Shows estimated audio duration for each slide.

If you need to fix a narration, edit `narrations.json` directly in Google Drive, then re-run Cell 5 with `REDO_SLIDES` set to the slide numbers you changed.

In [ ]:
import json

if not NARRATIONS_FILE.exists():
    print('❌ narrations.json not found — run Cell 3 first.')
else:
    with open(NARRATIONS_FILE, 'r', encoding='utf-8') as f:
        narrations = json.load(f)

    total_words = 0
    failed = []

    print(f'Narrations for: {PPTX_FILENAME}')
    print(f'Total slides  : {len(narrations)}')
    print('=' * 70)

    for key, data in sorted(narrations.items(), key=lambda x: x[1]['slide_number']):
        num   = data['slide_number']
        title = data['title']
        text  = data['narration']

        if text.startswith('[GENERATION FAILED'):
            print(f'\nSlide {num:02d}: {title}')
            print(f'  ❌ FAILED — re-run Cell 3 to retry')
            failed.append(num)
            continue

        words = len(text.split())
        est_s = int(words / 2.5)
        total_words += words

        print(f'\nSlide {num:02d}: {title}')
        print(f'  [{words} words, ~{est_s}s]')
        print(f'  {text[:300]}{"..." if len(text) > 300 else ""}')
        print('-' * 70)

    total_est = int(total_words / 2.5)
    print(f'\nTotal estimated audio: ~{total_est//60}m {total_est%60}s')
    if failed:
        print(f'Failed slides: {failed} — re-run Cell 3 to regenerate')

---
## 🔊 Cell 5 — Step 2: Generate WAV Audio Files (Google Cloud TTS)

This cell reads `narrations.json` and generates a WAV audio file for each slide using Google Cloud Text-to-Speech.

**Output:** `slide_01.wav`, `slide_02.wav`, ... saved to your presentation's output folder.

**Re-generating specific slides:**
If you edited some narrations in `narrations.json`, set `REDO_SLIDES` to the slide numbers you changed and re-run this cell. Only those slides will be regenerated — all others are skipped.

```python
REDO_SLIDES = [3, 7, 12]  # re-generate slides 3, 7, and 12
REDO_SLIDES = []           # generate all slides (skip existing WAV files)
```

**Cost estimate:** Google Studio voices cost approximately $0.16 per 1 million characters. A typical 20-slide presentation costs less than $0.01.

**Troubleshooting:**
- If you get an authentication error, check that `google_tts_credentials.json` is in the credentials folder and Cell 2 has been run
- If a voice name is invalid, check the full list at https://cloud.google.com/text-to-speech/docs/voices

In [ ]:
import json
from google.cloud import texttospeech

# ── Set slide numbers to re-generate (leave empty to generate all new slides) ──
# Example: REDO_SLIDES = [3, 7, 12]  to regenerate slides 3, 7 and 12
# Example: REDO_SLIDES = []          to skip slides that already have a WAV file
REDO_SLIDES = []

# ── Load narrations ────────────────────────────────────────────────────────────
if not NARRATIONS_FILE.exists():
    print('❌ narrations.json not found — run Cell 3 first.')
else:
    with open(NARRATIONS_FILE, 'r', encoding='utf-8') as f:
        narrations = json.load(f)

    # ── Cost estimate ──────────────────────────────────────────────────────────
    valid = {k: v for k, v in narrations.items()
             if not v['narration'].startswith('[GENERATION FAILED')}
    total_chars = sum(len(v['narration']) for v in valid.values())
    est_cost    = (total_chars / 1_000_000) * 160  # Studio voice pricing
    print(f'Presentation : {PPTX_FILENAME}')
    print(f'Slides       : {len(narrations)} ({len(valid)} valid, {len(narrations)-len(valid)} failed)')
    print(f'Total chars  : {total_chars:,}')
    print(f'Est. cost    : ~${est_cost:.4f} USD')
    print(f'Voice        : {VOICE_NAME} (rate={SPEAKING_RATE}, pitch={PITCH})')
    print()

    # ── TTS client and voice settings ─────────────────────────────────────────
    client = texttospeech.TextToSpeechClient()

    voice = texttospeech.VoiceSelectionParams(
        language_code = 'en-US',
        name          = VOICE_NAME,
    )

    audio_config = texttospeech.AudioConfig(
        audio_encoding    = texttospeech.AudioEncoding.LINEAR16,  # WAV format
        sample_rate_hertz = 24000,     # 24kHz — maximum for Studio voices
        speaking_rate     = SPEAKING_RATE,
        pitch             = PITCH,
    )

    # ── Synthesize each slide ──────────────────────────────────────────────────
    total     = len(narrations)
    generated = 0
    skipped   = 0
    failed    = 0

    for key, data in sorted(narrations.items(), key=lambda x: x[1]['slide_number']):
        slide_num = data['slide_number']
        narration = data['narration']
        wav_path  = SLIDE_OUTPUT_DIR / f'slide_{slide_num:02d}.wav'

        # Skip slides where narration generation failed
        if narration.startswith('[GENERATION FAILED'):
            print(f'  [{slide_num:02d}/{total}] SKIPPED — narration failed, re-run Cell 3 to fix')
            failed += 1
            continue

        # Skip or redo logic
        if REDO_SLIDES:
            if slide_num not in REDO_SLIDES:
                print(f'  [{slide_num:02d}/{total}] Skipped (not in REDO_SLIDES)')
                skipped += 1
                continue
        else:
            if wav_path.exists():
                print(f'  [{slide_num:02d}/{total}] Skipped — {wav_path.name} already exists')
                skipped += 1
                continue

        print(f'  [{slide_num:02d}/{total}] Synthesizing: {data["title"][:50]}', end='', flush=True)

        try:
            response = client.synthesize_speech(
                input        = texttospeech.SynthesisInput(text=narration),
                voice        = voice,
                audio_config = audio_config,
            )
            wav_path.write_bytes(response.audio_content)
            size_kb = wav_path.stat().st_size // 1024
            print(f' ✓  ({size_kb} KB)')
            generated += 1

        except Exception as e:
            print(f'\n       ✗ ERROR: {e}')
            failed += 1

    print(f'\n✅ Step 2 complete!')
    print(f'   Generated : {generated} WAV files')
    print(f'   Skipped   : {skipped}')
    print(f'   Failed    : {failed}')
    print(f'   Output    : {SLIDE_OUTPUT_DIR}')
    print()
    print('▶  Run Cell 6 to embed audio into the PowerPoint file.')

---
## 🎬 Cell 6 — Step 3: Embed Audio into PowerPoint

This cell embeds each WAV file directly into its corresponding slide in the PowerPoint file. Each slide is configured to autoplay its audio when the slideshow reaches that slide.

**Output:** `<PresentationName>_narrated.pptx` saved to your output folder.

**The original PowerPoint file is never modified** — a new copy is always created.

**After this cell completes:**
1. Download `_narrated.pptx` from your Drive output folder
2. Open it in Microsoft PowerPoint
3. Go to **File → Export → Create a Video**
4. In the dropdown, select **"Use Recorded Timings and Narrations"**
5. Choose your video quality (1080p recommended) and click **Create Video**

**Technical notes** (for reference — no action required):
- Audio is embedded using the same XML structure PowerPoint generates when you use Insert → Audio → Audio on My PC
- Each slide requires two audio relationships: a standard OOXML `audio` relationship and a Microsoft-specific `media` relationship
- Slide advance timing is set programmatically to match the exact duration of each WAV file
- The `<p:timing>` element is injected as a direct child of `<p:sld>` (after `</p:cSld>`), which is required by PowerPoint's XML schema

In [ ]:
import zipfile, re, wave, base64
from lxml import etree
from pptx import Presentation
from pathlib import Path

# ── Namespaces ─────────────────────────────────────────────────────────────────
REL_NS         = 'http://schemas.openxmlformats.org/package/2006/relationships'
CT_NS          = 'http://schemas.openxmlformats.org/package/2006/content-types'
AUDIO_REL_TYPE = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/audio'
MEDIA_REL_TYPE = 'http://schemas.microsoft.com/office/2007/relationships/media'
IMAGE_REL_TYPE = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships/image'
PNS = 'http://schemas.openxmlformats.org/presentationml/2006/main'
ANS = 'http://schemas.openxmlformats.org/drawingml/2006/main'
RNS = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'

# ── Speaker icon ───────────────────────────────────────────────────────────────
# A valid 1x1 white pixel PNG is required as the audio shape's image reference.
# PowerPoint silently hides any p:pic shape with an empty or missing blip image.
SPEAKER_ICON_B64   = 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwADhQGAWjR9awAAAABJRU5ErkJggg=='
SPEAKER_ICON_BYTES = base64.b64decode(SPEAKER_ICON_B64)
SPEAKER_ICON_PATH  = 'ppt/media/speaker_icon.png'


def get_wav_duration_ms(wav_path):
    """Return WAV duration in milliseconds for slide timing."""
    try:
        with wave.open(str(wav_path), 'r') as w:
            return int(w.getnframes() / w.getframerate() * 1000)
    except Exception:
        return 60000  # fallback: 60 seconds


def get_max_shape_id(slide_bytes):
    """Find the highest shape ID in slide XML to avoid collisions."""
    ids = [int(x) for x in re.findall(rb'<[^>]+ id="(\d+)"', slide_bytes)]
    return max(ids) if ids else 100


def make_shape(rId_link, rId_embed, rId_icon, shape_id):
    """
    Build the audio shape XML.
    Replicates exactly what PowerPoint generates via Insert -> Audio -> Audio on My PC.
    Requires two audio relationships (rId_link and rId_embed) and a valid image blip.
    """
    return (
        f'<p:pic xmlns:p="{PNS}" xmlns:a="{ANS}" xmlns:r="{RNS}">'
          f'<p:nvPicPr>'
            f'<p:cNvPr id="{shape_id}" name="Audio_{shape_id}">'
              f'<a:hlinkClick r:id="" action="ppaction://media"/>'
            f'</p:cNvPr>'
            f'<p:cNvPicPr><a:picLocks noChangeAspect="1"/></p:cNvPicPr>'
            f'<p:nvPr>'
              f'<a:audioFile r:link="{rId_link}"/>'
              f'<p:extLst>'
                f'<p:ext uri="{{DAA4B4D4-6D71-4841-9C94-3DE7FCFB9230}}">'
                  f'<p14:media xmlns:p14="http://schemas.microsoft.com/office/powerpoint/2010/main"'
                  f' r:embed="{rId_embed}"/>'
                f'</p:ext>'
              f'</p:extLst>'
            f'</p:nvPr>'
          f'</p:nvPicPr>'
          f'<p:blipFill>'
            f'<a:blip r:embed="{rId_icon}"/>'
            f'<a:stretch><a:fillRect/></a:stretch>'
          f'</p:blipFill>'
          f'<p:spPr>'
            f'<a:xfrm><a:off x="457200" y="6400000"/><a:ext cx="304800" cy="304800"/></a:xfrm>'
            f'<a:prstGeom prst="rect"><a:avLst/></a:prstGeom>'
          f'</p:spPr>'
        f'</p:pic>'
    ).encode('utf-8')


def make_timing(shape_id, dur_ms):
    """
    Build the animation timing XML for autoplay audio.
    delay='0' on all stCondLst entries = autoplay on slide entry.
    p:audio/p:cMediaNode is required for PowerPoint to register this as media.
    Must be a direct child of p:sld (injected after </p:cSld>).
    """
    return (
        f'<p:timing xmlns:p="{PNS}" xmlns:a="{ANS}">'
          f'<p:tnLst><p:par>'
            f'<p:cTn id="1" dur="indefinite" restart="never" nodeType="tmRoot">'
              f'<p:childTnLst>'
                f'<p:seq concurrent="1" nextAc="seek">'
                  f'<p:cTn id="2" dur="indefinite" nodeType="mainSeq">'
                    f'<p:childTnLst><p:par>'
                      f'<p:cTn id="3" fill="hold">'
                        f'<p:stCondLst><p:cond delay="0"/></p:stCondLst>'
                        f'<p:childTnLst><p:par>'
                          f'<p:cTn id="4" fill="hold">'
                            f'<p:stCondLst><p:cond delay="0"/></p:stCondLst>'
                            f'<p:childTnLst><p:par>'
                              f'<p:cTn id="5" presetID="1" presetClass="mediacall" presetSubtype="0" fill="hold" nodeType="clickEffect">'
                                f'<p:stCondLst><p:cond delay="0"/></p:stCondLst>'
                                f'<p:childTnLst>'
                                  f'<p:cmd type="call" cmd="playFrom(0.0)">'
                                    f'<p:cBhvr>'
                                      f'<p:cTn id="6" dur="{dur_ms}" fill="hold"/>'
                                      f'<p:tgtEl><p:spTgt spid="{shape_id}"/></p:tgtEl>'
                                    f'</p:cBhvr>'
                                  f'</p:cmd>'
                                f'</p:childTnLst>'
                              f'</p:cTn>'
                            f'</p:par></p:childTnLst>'
                          f'</p:cTn>'
                        f'</p:par></p:childTnLst>'
                      f'</p:cTn>'
                    f'</p:par></p:childTnLst>'
                  f'</p:cTn>'
                  f'<p:prevCondLst>'
                    f'<p:cond evt="onPrev" delay="0"><p:tgtEl><p:sldTgt/></p:tgtEl></p:cond>'
                  f'</p:prevCondLst>'
                  f'<p:nextCondLst>'
                    f'<p:cond evt="onNext" delay="0"><p:tgtEl><p:sldTgt/></p:tgtEl></p:cond>'
                  f'</p:nextCondLst>'
                f'</p:seq>'
                f'<p:audio>'
                  f'<p:cMediaNode vol="80000">'
                    f'<p:cTn id="7" fill="hold" display="0">'
                      f'<p:stCondLst><p:cond delay="0"/></p:stCondLst>'
                      f'<p:endCondLst>'
                        f'<p:cond evt="onStopAudio" delay="0"><p:tgtEl><p:sldTgt/></p:tgtEl></p:cond>'
                      f'</p:endCondLst>'
                    f'</p:cTn>'
                    f'<p:tgtEl><p:spTgt spid="{shape_id}"/></p:tgtEl>'
                  f'</p:cMediaNode>'
                f'</p:audio>'
              f'</p:childTnLst>'
            f'</p:cTn>'
          f'</p:par></p:tnLst>'
        f'</p:timing>'
    ).encode('utf-8')


def make_transition(dur_ms):
    """
    Set slide auto-advance timing.
    Required for PowerPoint's File -> Export -> Create a Video
    to recognise timings and enable 'Use Recorded Timings and Narrations'.
    advAuto='1' = advance automatically; advTm = advance time in milliseconds.
    """
    return (
        f'<p:transition xmlns:p="{PNS}" advAuto="1" advTm="{dur_ms}"/>'
    ).encode('utf-8')


def inject_slide(slide_bytes, shape_bytes, timing_bytes, transition_bytes):
    """
    Inject audio shape, transition and timing into raw slide XML bytes.

    IMPORTANT: Uses targeted byte replacement, NOT full etree parse/reserialize.
    Reserializing the full slide XML corrupts namespace declarations that
    PowerPoint requires, causing the file to fail to open.

    Injection order (required by PowerPoint XML schema):
    1. Audio shape  → before </p:spTree>
    2. Transition   → after </p:cSld>  (direct child of p:sld)
    3. Timing       → after transition (direct child of p:sld)

    Critical: inject after </p:cSld>, NOT before <p:extLst>.
    <p:extLst> also appears inside individual shapes, so targeting it
    causes timing to be injected inside a shape instead of at slide level.
    """
    # 1. Inject audio shape before </p:spTree>
    slide_bytes = slide_bytes.replace(b'</p:spTree>', shape_bytes + b'</p:spTree>', 1)

    # 2. Remove any existing transition and timing blocks
    slide_bytes = re.sub(rb'<p:transition\b[^/]*/>', b'', slide_bytes)
    slide_bytes = re.sub(rb'<p:transition\b.*?</p:transition>', b'', slide_bytes, flags=re.DOTALL)
    slide_bytes = re.sub(rb'<p:timing\b.*?</p:timing>', b'', slide_bytes, flags=re.DOTALL)

    # 3. Inject transition then timing after </p:cSld>
    slide_bytes = slide_bytes.replace(
        b'</p:cSld>',
        b'</p:cSld>' + transition_bytes + timing_bytes,
        1
    )

    return slide_bytes


def embed_audio_in_pptx(pptx_path, audio_dir, output_path):
    """Embed WAV audio into each slide of a PPTX with autoplay timing."""

    prs = Presentation(pptx_path)
    total = len(prs.slides)
    slide_partnames = [slide.part.partname for slide in prs.slides]
    del prs

    print(f'Processing : {pptx_path.name}  ({total} slides)')
    print(f'Audio dir  : {audio_dir}\n')

    # Read the entire PPTX into memory as a dict of zip entries
    with zipfile.ZipFile(pptx_path, 'r') as zin:
        content = {name: zin.read(name) for name in zin.namelist()}

    # Add speaker icon PNG — used as blip image for all audio shapes
    content[SPEAKER_ICON_PATH] = SPEAKER_ICON_BYTES

    embedded = 0
    skipped  = 0

    for i, partname in enumerate(slide_partnames, start=1):
        wav_path = audio_dir / f'slide_{i:02d}.wav'

        if not wav_path.exists():
            print(f'  [{i:02d}/{total}] ⚠️  No audio: slide_{i:02d}.wav not found — skipping')
            skipped += 1
            continue

        print(f'  [{i:02d}/{total}] Embedding slide_{i:02d}.wav ...', end='', flush=True)

        try:
            audio_bytes = wav_path.read_bytes()
            dur_ms      = get_wav_duration_ms(wav_path)
            slide_zip   = partname.lstrip('/')
            slide_dir   = '/'.join(slide_zip.split('/')[:-1])
            slide_file  = slide_zip.split('/')[-1]
            rels_zip    = f'{slide_dir}/_rels/{slide_file}.rels'
            media_zip   = f'ppt/media/audio_slide{i:02d}.wav'
            rId_link    = f'rIdAudioLink{i:02d}'
            rId_embed   = f'rIdAudioEmbed{i:02d}'
            rId_icon    = f'rIdAudioIcon{i:02d}'

            # 1. Add WAV audio bytes to the zip content dict
            content[media_zip] = audio_bytes

            # 2. Register three relationships in the slide's .rels file:
            #    rId_link  → standard OOXML audio relationship (used by audioFile r:link)
            #    rId_embed → Microsoft-specific media relationship (required for playback)
            #    rId_icon  → image relationship for the speaker icon blip
            rels_tree = etree.fromstring(content[rels_zip])
            existing  = {el.get('Id') for el in rels_tree}
            for rid, rtype, target in [
                (rId_link,  AUDIO_REL_TYPE, f'../media/audio_slide{i:02d}.wav'),
                (rId_embed, MEDIA_REL_TYPE, f'../media/audio_slide{i:02d}.wav'),
                (rId_icon,  IMAGE_REL_TYPE, '../media/speaker_icon.png'),
            ]:
                if rid not in existing:
                    el = etree.SubElement(rels_tree, f'{{{REL_NS}}}Relationship')
                    el.set('Id', rid)
                    el.set('Type', rtype)
                    el.set('Target', target)
            content[rels_zip] = etree.tostring(
                rels_tree, xml_declaration=True, encoding='UTF-8', standalone=True
            )

            # 3. Build XML components and inject into raw slide bytes
            shape_id         = get_max_shape_id(content[slide_zip]) + 1
            shape_bytes      = make_shape(rId_link, rId_embed, rId_icon, shape_id)
            timing_bytes     = make_timing(shape_id, dur_ms)
            transition_bytes = make_transition(dur_ms)
            content[slide_zip] = inject_slide(
                content[slide_zip], shape_bytes, timing_bytes, transition_bytes
            )

            # 4. Register WAV and PNG content types if not already present
            ct_tree = etree.fromstring(content['[Content_Types].xml'])
            existing_exts = {el.get('Extension') for el in ct_tree.findall(f'{{{CT_NS}}}Default')}
            for ext, ctype in [('wav', 'audio/wav'), ('png', 'image/png')]:
                if ext not in existing_exts:
                    ov = etree.SubElement(ct_tree, f'{{{CT_NS}}}Default')
                    ov.set('Extension', ext)
                    ov.set('ContentType', ctype)
            content['[Content_Types].xml'] = etree.tostring(
                ct_tree, xml_declaration=True, encoding='UTF-8', standalone=True
            )

            size_kb = len(audio_bytes) // 1024
            print(f' ✓  ({size_kb} KB, {dur_ms//1000}s)')
            embedded += 1

        except Exception as e:
            import traceback
            print(f'\n       ✗ ERROR on slide {i}: {e}')
            traceback.print_exc()
            skipped += 1

    # Write the modified content back as a new PPTX zip file
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zout:
        for name, data in content.items():
            zout.writestr(name, data)

    print(f'\n✅ Step 3 complete!')
    print(f'   Slides with audio : {embedded}')
    print(f'   Slides skipped    : {skipped}')
    print(f'   Output saved to   : {output_path}')
    print()
    print('▶  Next steps in PowerPoint:')
    print('   1. Download the _narrated.pptx file from your Drive')
    print('   2. Open in Microsoft PowerPoint')
    print('   3. File → Export → Create a Video')
    print('   4. Select "Use Recorded Timings and Narrations"')
    print('   5. Choose resolution (1080p recommended) → Create Video')


# ── Run ────────────────────────────────────────────────────────────────────────
narrated_output = SLIDE_OUTPUT_DIR / f'{PRESENTATION_NAME}_narrated.pptx'

embed_audio_in_pptx(
    pptx_path   = PPTX_PATH,
    audio_dir   = SLIDE_OUTPUT_DIR,
    output_path = narrated_output
)